# Model Training

Training 4 models: Logistic Regression, Random Forest, XGBoost, LightGBM. But first, let's reproduce the original broken model to see how bad it really was, then try different approaches for handling the class imbalance.

In [7]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, recall_score)
from src.data_loader import load_raw_data, get_feature_target
from src.preprocessing import split_data, impute_missing, scale_features
from src.feature_engineering import create_interaction_features
from src.models import get_model_configs, train_models, get_cv
from src.config import CONFIG

sns.set_theme(style='whitegrid')
%matplotlib inline

In [8]:
# load and preprocess — no leakage this time
df = load_raw_data()
X, y = get_feature_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)
X_train, X_test, imputer = impute_missing(X_train, X_test)

# add interaction features
X_train_full = create_interaction_features(X_train)
X_test_full = create_interaction_features(X_test)

# scaled for logistic regression
X_train_sc, X_test_sc, scaler = scale_features(X_train_full, X_test_full)

print(f"Train: {X_train_full.shape} | Test: {X_test_full.shape}")
print(f"Train target: {y_train.value_counts().to_dict()}")
print(f"Test target:  {y_test.value_counts().to_dict()}")

Train: (2400, 34) | Test: (600, 34)
Train target: {0: 2000, 1: 400}
Test target:  {0: 500, 1: 100}


## Baseline — the original (broken) approach

Default logistic regression, no imbalance handling. Let's see what we're up against.

In [9]:
baseline = LogisticRegression(max_iter=1000, random_state=CONFIG['random_state'])
baseline.fit(X_train_sc, y_train)
y_pred_base = baseline.predict(X_test_sc)
y_prob_base = baseline.predict_proba(X_test_sc)[:, 1]

print("BASELINE (default LogReg, no imbalance handling)")
print("=" * 50)
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_base)}")
print(f"\n{classification_report(y_test, y_pred_base, target_names=['Good', 'Bad'])}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"Bad loan recall: {recall_score(y_test, y_pred_base):.1%}")
print(f"\nYeah... catching {recall_score(y_test, y_pred_base):.0%} of bad loans is not gonna work.")

BASELINE (default LogReg, no imbalance handling)
Confusion Matrix:
[[478  22]
 [ 82  18]]

              precision    recall  f1-score   support

        Good       0.85      0.96      0.90       500
         Bad       0.45      0.18      0.26       100

    accuracy                           0.83       600
   macro avg       0.65      0.57      0.58       600
weighted avg       0.79      0.83      0.79       600

ROC-AUC: 0.7587
Bad loan recall: 18.0%

Yeah... catching 18% of bad loans is not gonna work.


## Dealing with the imbalance

83% good / 17% bad. Let's try a few approaches and see which works best.

In [10]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG['random_state'])

# 1. no handling
lr_none = LogisticRegression(max_iter=1000, random_state=CONFIG['random_state'])
scores_none = cross_val_score(lr_none, X_train_sc, y_train, cv=cv, scoring='roc_auc')

# 2. class weights
lr_balanced = LogisticRegression(max_iter=1000, random_state=CONFIG['random_state'],
                                 class_weight='balanced')
scores_balanced = cross_val_score(lr_balanced, X_train_sc, y_train, cv=cv, scoring='roc_auc')

# 3. SMOTE
smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=CONFIG['random_state'])),
    ('lr', LogisticRegression(max_iter=1000, random_state=CONFIG['random_state']))
])
scores_smote = cross_val_score(smote_pipe, X_train_sc, y_train, cv=cv, scoring='roc_auc')

# 4. cost-sensitive (5:1 reflecting the $500/$100 ratio)
sample_weights = np.where(y_train == 1, 5.0, 1.0)
lr_cost = LogisticRegression(max_iter=1000, random_state=CONFIG['random_state'])
scores_cost = []
for train_idx, val_idx in cv.split(X_train_sc, y_train):
    Xtr, Xval = X_train_sc.iloc[train_idx], X_train_sc.iloc[val_idx]
    ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]
    lr_cost.fit(Xtr, ytr, sample_weight=sample_weights[train_idx])
    scores_cost.append(roc_auc_score(yval, lr_cost.predict_proba(Xval)[:, 1]))
scores_cost = np.array(scores_cost)

results = pd.DataFrame({
    'Approach': ['No handling', 'Class weights', 'SMOTE', 'Cost-sensitive (5:1)'],
    'AUC_mean': [scores_none.mean(), scores_balanced.mean(),
                 scores_smote.mean(), scores_cost.mean()],
    'AUC_std': [scores_none.std(), scores_balanced.std(),
                scores_smote.std(), scores_cost.std()],
})
results['ROC-AUC'] = results.apply(lambda r: f"{r['AUC_mean']:.4f} +/- {r['AUC_std']:.4f}", axis=1)

print("IMBALANCE STRATEGY COMPARISON (LogReg, 5-fold CV)")
print("=" * 55)
results[['Approach', 'ROC-AUC']]

IMBALANCE STRATEGY COMPARISON (LogReg, 5-fold CV)


,Approach,ROC-AUC
0,No handling,0.7962 +/- 0.0216
1,Class weights,0.7946 +/- 0.0220
2,SMOTE,0.7919 +/- 0.0246
3,Cost-sensitive (5:1),0.7943 +/- 0.0220


In [11]:
# more importantly — how does recall change?
print("Recall on bad loans (threshold=0.5):")
for name, m in [('No handling', lr_none), ('Class weights', lr_balanced)]:
    m.fit(X_train_sc, y_train)
    print(f"  {name:25s} {recall_score(y_test, m.predict(X_test_sc)):.1%}")

smote = SMOTE(random_state=CONFIG['random_state'])
X_sm, y_sm = smote.fit_resample(X_train_sc, y_train)
lr_smote = LogisticRegression(max_iter=1000, random_state=CONFIG['random_state'])
lr_smote.fit(X_sm, y_sm)
print(f"  {'SMOTE':25s} {recall_score(y_test, lr_smote.predict(X_test_sc)):.1%}")

lr_cost.fit(X_train_sc, y_train, sample_weight=sample_weights)
print(f"  {'Cost-sensitive':25s} {recall_score(y_test, lr_cost.predict(X_test_sc)):.1%}")

print(f"\nSMOTE: {len(X_sm):,} samples (was {len(X_train_sc):,})")
print(f"  Good: {sum(y_sm==0):,} | Bad: {sum(y_sm==1):,}")

Recall on bad loans (threshold=0.5):
  No handling               18.0%
  Class weights             62.0%
  SMOTE                     63.0%
  Cost-sensitive            62.0%

SMOTE: 4,000 samples (was 2,400)
  Good: 2,000 | Bad: 2,000


Class weights and SMOTE both help a lot. Let's proceed with class weights baked into each model (simpler than SMOTE and works about as well).

## Training all models

RandomizedSearchCV with 5-fold stratified CV. This takes a couple minutes.

In [12]:
model_configs = get_model_configs()

# LogReg needs scaled features, tree models don't
print("=" * 60)
print("TRAINING (RandomizedSearchCV, 5-fold stratified CV)")
print("=" * 60)

lr_configs = {"Logistic Regression": model_configs["Logistic Regression"]}
lr_results = train_models(X_train_sc, y_train, lr_configs)

tree_configs = {k: v for k, v in model_configs.items() if k != "Logistic Regression"}
tree_results = train_models(X_train_full, y_train, tree_configs)

all_results = {**lr_results, **tree_results}

TRAINING (RandomizedSearchCV, 5-fold stratified CV)

Training Logistic Regression...


e:\conda\miniconda3\envs\credit-risk\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  Best ROC-AUC: 0.7993
  Best params: {'penalty': 'l2', 'class_weight': None, 'C': 0.1}

Training Random Forest...
  Best ROC-AUC: 0.7931
  Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 20, 'max_depth': 7, 'class_weight': 'balanced_subsample'}

Training XGBoost...


e:\conda\miniconda3\envs\credit-risk\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:48:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Best ROC-AUC: 0.7935
  Best params: {'subsample': 1.0, 'scale_pos_weight': 3, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

Training LightGBM...
  Best ROC-AUC: 0.7874
  Best params: {'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 1, 'num_leaves': 31, 'n_estimators': 200, 'max_depth': -1, 'learning_rate': 0.01, 'is_unbalance': True, 'colsample_bytree': 0.7}


In [14]:
# comparison table
comparison = []
for name, search in all_results.items():
    model = search.best_estimator_
    if name == "Logistic Regression":
        y_prob = model.predict_proba(X_test_sc)[:, 1]
    else:
        y_prob = model.predict_proba(X_test_full)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    comparison.append({
        'Model': name,
        'CV AUC': f"{search.best_score_:.4f} +/- {search.cv_results_['std_test_score'][search.best_index_]:.4f}",
        'Test AUC': f"{roc_auc_score(y_test, y_prob):.4f}",
        'Test PR-AUC': f"{average_precision_score(y_test, y_prob):.4f}",
        'Bad Recall': f"{recall_score(y_test, y_pred):.1%}",
    })

print("\nMODEL COMPARISON")
print("=" * 80)
pd.DataFrame(comparison)


MODEL COMPARISON


,Model,CV AUC,Test AUC,Test PR-AUC,Bad Recall
0,Logistic Regression,0.7993 +/- 0.0208,0.7623,0.3955,17.0%
1,Random Forest,0.7931 +/- 0.0212,0.7868,0.4149,64.0%
2,XGBoost,0.7935 +/- 0.0197,0.7813,0.4243,52.0%
3,LightGBM,0.7874 +/- 0.0230,0.7714,0.4153,53.0%


In [15]:
# detailed reports
for name, search in all_results.items():
    model = search.best_estimator_
    y_pred = model.predict(X_test_sc if name == "Logistic Regression" else X_test_full)
    print(f"\n--- {name} ---")
    print(f"Best params: {search.best_params_}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=['Good', 'Bad']))


--- Logistic Regression ---
Best params: {'penalty': 'l2', 'class_weight': None, 'C': 0.1}
[[481  19]
 [ 83  17]]
              precision    recall  f1-score   support

        Good       0.85      0.96      0.90       500
         Bad       0.47      0.17      0.25       100

    accuracy                           0.83       600
   macro avg       0.66      0.57      0.58       600
weighted avg       0.79      0.83      0.80       600


--- Random Forest ---
Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 20, 'max_depth': 7, 'class_weight': 'balanced_subsample'}
[[401  99]
 [ 36  64]]
              precision    recall  f1-score   support

        Good       0.92      0.80      0.86       500
         Bad       0.39      0.64      0.49       100

    accuracy                           0.78       600
   macro avg       0.66      0.72      0.67       600
weighted avg       0.83      0.78      0.79       600


--- XGBoost ---
Best params: {'subsample': 1.0

In [16]:
# save models
import os
os.makedirs('../data/processed', exist_ok=True)

for name, search in all_results.items():
    fname = name.lower().replace(' ', '_')
    joblib.dump(search.best_estimator_, f'../data/processed/model_{fname}.joblib')

joblib.dump(baseline, '../data/processed/model_baseline.joblib')
print("Models saved.")

Models saved.


Big improvement over the baseline. The tree models do well on AUC but the real test is in the next notebook where we optimize the threshold for profit.